# NONLINEAR_MODELS

XGBoost, SVR and Random Forest (Section 3.4), each as a baseline (default hyperparams) and a tuned model, plus the dummy regressor. NNs are trained in the colab notebook and write their results in the same format.

- Splits: original 70/15/15 by row index.
- Tuning: each candidate is fit on the training rows and scored on the val rows (holdout), the best val R^2 wins. RMSE and R^2 are both reported.
- TSCV: original 24 h/12 h forward chaining on the first 85% with the selected hyperparams (no tuning inside TSCV). This is just for a robustness check only.
- Test predictions are written at the same time ONLY for convenience, to a separate folder/file. Model selection (here and in STATIC_SIMULATION) NEVER reads them.

## Imports

In [13]:
import json
import math
import pickle
import time
from pathlib import Path

import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

## Configuration

In [14]:
MANIFEST_PATH = Path("data/processed/manifest.json")
RESULTS_DIR = Path("results/nonlinear")
PREDICTION_DIR = Path("results/predictions") # validation/ and test/ (shared with colab NN notebook)
MODELS_PATH = Path("models/revised")
IMAGES_PATH = Path("images/nonlinear")
LOCKED_SELECTION_PATH = Path("results/simulation/locked_selection.json")

RANDOM_SEED = 42
N_JOBS = -1
SAMPLINGS = None # None = all sampling rates
TARGETS = None # None = all lead0 drift targets (derivative targets are only GLMs)
MODELS_TO_RUN = ["dummy_baseline", "xgb_baseline", "xgb_tuned", "svr_baseline", "svr_tuned", "rf_baseline", "rf_tuned"]
TSCV_MODELS = ["xgb_baseline", "xgb_tuned", "svr_baseline", "svr_tuned", "rf_baseline", "rf_tuned"]
SKIP_FINISHED = True # skip configs where results already exist (lets a long run resume--super convenient)
PLOT_TRAIN_VAL = False # one fig per config
CI_LEVEL = 0.95

# Lead time runs (L = 2/5/10 min) for the pipeline locked in STATIC_SIMULATION, or a list of {"sampling": .., "target": <lead0 target>, "model": ...}
RUN_LEAD_TIMES = False
LEAD_TIME_PIPELINES = "locked"
LEAD_TIMES_S = [120, 300, 600]

## Hyperparameters

Baselines and search spaces are the ones from original notebooks.

In [15]:
BASELINE_PARAMS = {
    "xgb": {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 6, "subsample": 0.8, "colsample_bytree": 0.8, "reg_alpha": 0.0, "reg_lambda": 1.0, "min_child_weight": 1, "gamma": 0.0},
    "svr": {"C": 10.0, "epsilon": 0.1, "gamma": "scale"},
    "rf": {"n_estimators": 500, "max_depth": None},
}
XGB_FIXED = {"objective": "reg:squarederror", "random_state": RANDOM_SEED, "n_jobs": N_JOBS, "tree_method": "hist"}
RF_FIXED = {"random_state": RANDOM_SEED, "n_jobs": N_JOBS, "bootstrap": True}

XGB_N_TRIALS = 500
XGB_SPACE = {
    "n_estimators": lambda r: int(r.choice([200, 400, 600, 800, 1000, 1200])),
    "learning_rate": lambda r: float(10 ** r.uniform(-2.0, -0.3)),
    "max_depth": lambda r: int(r.integers(3, 13)),
    "subsample": lambda r: float(r.uniform(0.6, 1.0)),
    "colsample_bytree": lambda r: float(r.uniform(0.6, 1.0)),
    "reg_alpha": lambda r: float(10 ** r.uniform(-6, 1)),
    "reg_lambda": lambda r: float(10 ** r.uniform(-3, 2)),
    "min_child_weight": lambda r: int(r.integers(1, 11)),
    "gamma": lambda r: float(r.uniform(0.0, 5.0)),
}

SVR_N_COARSE, SVR_N_REFINE = 60, 40


def svr_coarse(r):
    g = r.uniform()
    gamma = "scale" if g < 0.34 else ("auto" if g < 0.68 else float(10 ** r.uniform(-4, 0)))
    return {"C": float(10 ** r.uniform(-2, 3)), "epsilon": float(10 ** r.uniform(-3, -0.3)), "gamma": gamma}


def svr_refine(r, best):
    g = best["gamma"]
    gamma = g * 10 ** r.uniform(-0.5, 0.5) if not isinstance(g, str) else (g if r.uniform() < 0.5 else ("scale" if g == "auto" else "auto"))
    return {"C": best["C"] * 10 ** r.uniform(-0.5, 0.5), "epsilon": best["epsilon"] * 10 ** r.uniform(-0.5, 0.5), "gamma": gamma}


RF_GRID = {"n_estimators": [200, 400, 600, 800, 1000], "max_depth": [None, 3, 4, 6, 8, 10, 12, 16, 20]}

In [16]:
mpl.rc("axes", labelsize=14)
mpl.rc("xtick", labelsize=12)
mpl.rc("ytick", labelsize=12)

for d in (RESULTS_DIR / "runs", RESULTS_DIR / "tscv", MODELS_PATH, IMAGES_PATH, PREDICTION_DIR / "validation", PREDICTION_DIR / "test"):
    d.mkdir(parents=True, exist_ok=True)

manifest = json.loads(MANIFEST_PATH.read_text())
FEATURES = manifest["feature_columns"]
TARGET_META = {t["name"]: t for t in manifest["targets"]}
SAMPLING_S = manifest["sampling_s"]
SAMPLINGS = SAMPLINGS or list(SAMPLING_S)
TARGETS = TARGETS or [t["name"] for t in manifest["targets"] if t["primary"] and not t["glm_only"]]
TZ = manifest["display_timezone"]

## Splits, metrics, prediction files

In [17]:
def load_dataset(sampling):
    df = pd.read_csv(manifest["files"]["datasets"][sampling])
    assert len(df) == manifest["splits"][sampling]["n"], "dataset does not match manifest -- rerun DATA_PROCESSING"
    return df


def label_end(df, target):
    return df["current_time"].to_numpy() + TARGET_META[target]["window_end_offset_s"] * 1000


def split_masks(df, sampling, target):
    """70/15/15 by row. Rows whose label window runs into the next split are dropped."""
    sp = manifest["splits"][sampling]
    idx = np.arange(len(df))
    end = label_end(df, target)
    has = df[target].notna().to_numpy()
    train = (idx < sp["n_train"]) & (end <= sp["validation_start_ms"]) & has
    val = (idx >= sp["n_train"]) & (idx < sp["n_train"] + sp["n_validation"]) & (end <= sp["test_start_ms"]) & has
    test = (idx >= sp["n_train"] + sp["n_validation"]) & has
    
    return train, val, test


def metrics(y, yhat):
    return {"rmse": float(np.sqrt(mean_squared_error(y, yhat))), "r2": float(r2_score(y, yhat))}


PRED_COLUMNS = ["current_time", "split", "sampling", "sampling_s", "target", "metric", "window_samples", "lead_s", "variant", "window_start_offset_s", "window_end_offset_s", "model", "model_family", "y_true", "y_pred"]


def prediction_frame(df, mask, yhat, split, sampling, target, model):
    m = TARGET_META[target]
    out = pd.DataFrame({"current_time": df.loc[mask, "current_time"].to_numpy(), "y_true": df.loc[mask, target].to_numpy(), "y_pred": np.asarray(yhat, float)})
    out["split"], out["sampling"], out["sampling_s"], out["target"], out["model"] = split, sampling, SAMPLING_S[sampling], target, model
    
    for k in ("metric", "window_samples", "lead_s", "variant", "window_start_offset_s", "window_end_offset_s"):
        out[k] = m[k]
        
    out["model_family"] = "baseline" if model == "dummy_baseline" else "nonlinear"
    
    return out[PRED_COLUMNS]


def run_id(sampling, target, model):
    return f"{sampling}__{target}__{model}"

## Models

Scaling:
- XGBoost/RF: none. Tree splits are thresholds on single features so feature scale has no effect.
- SVR: StandardScaler on X and y inside the pipeline. The RBF kernel works on distances (features need coomparable scales) and epsilon is in target units.
- Dummy: none

Every scaler is fit on the training rows of the fit it belongs to.

In [18]:
def build(model, params):
    family = model.split("_")[0]
    if family == "dummy":
        return DummyRegressor(strategy="mean")
    if family == "xgb":
        return XGBRegressor(**params, **XGB_FIXED)
    if family == "rf":
        return RandomForestRegressor(**params, **RF_FIXED)
    if family == "svr":
        pipe = Pipeline([("scaler", StandardScaler()), ("svr", SVR(kernel="rbf", **params))])
        return TransformedTargetRegressor(regressor=pipe, transformer=StandardScaler())
        
    raise ValueError(model)


def tune(model, X, y, train, val):
    """Holdout search: fit on training rows, keep the candidate with the best val R^2."""
    family = model.split("_")[0]
    rng = np.random.default_rng(RANDOM_SEED)
    if family == "xgb":
        candidates = [{k: f(rng) for k, f in XGB_SPACE.items()} for _ in range(XGB_N_TRIALS)]
    elif family == "rf":
        candidates = [{"n_estimators": n, "max_depth": d} for n in RF_GRID["n_estimators"] for d in RF_GRID["max_depth"]]
    else:
        candidates = [svr_coarse(rng) for _ in range(SVR_N_COARSE)]

    best = {"r2": -np.inf}

    def try_all(cands):
        for p in cands:
            est = build(model, p).fit(X[train], y[train])
            r2 = r2_score(y[val], est.predict(X[val]))
            if r2 > best["r2"]:
                best.update(r2=r2, params=p, estimator=est)

    try_all(candidates)
    
    if family == "svr":
        try_all([svr_refine(rng, best["params"]) for _ in range(SVR_N_REFINE)])
        
    return best["params"], best["estimator"], len(candidates) + (SVR_N_REFINE if family == "svr" else 0)

## Train, validate, and save test predictions

In [19]:
def fit_one(df, sampling, target, model):
    X, y = df[FEATURES].to_numpy(float), df[target].to_numpy(float)
    train, val, test = split_masks(df, sampling, target)
    t0 = time.time()
    
    if model.endswith("_tuned"):
        params, est, n_trials = tune(model, X, y, train, val)
    else:
        params, n_trials = BASELINE_PARAMS.get(model.split("_")[0], {}), 0
        est = build(model, params).fit(X[train], y[train])
        
    yhat = {s: est.predict(X[m]) for s, m in (("train", train), ("validation", val), ("test", test))}
    m_tr, m_va, m_te = metrics(y[train], yhat["train"]), metrics(y[val], yhat["validation"]), metrics(y[test], yhat["test"])

    rid = run_id(sampling, target, model)
    with open(MODELS_PATH / f"model_{sampling}_{target}_{model}.pkl", "wb") as f:
        pickle.dump({"estimator": est, "params": params, "feature_columns": FEATURES}, f)
        
    pd.concat([prediction_frame(df, train, yhat["train"], "train", sampling, target, model), prediction_frame(df, val, yhat["validation"], "validation", sampling, target, model)]).to_csv(PREDICTION_DIR / "validation" / f"{rid}.csv", index=False)
    prediction_frame(df, test, yhat["test"], "test", sampling, target, model).to_csv(PREDICTION_DIR / "test" / f"{rid}.csv", index=False)

    record = {"sampling frequency": sampling, "model_name": model, "target_name": target, "params": json.dumps(params), "n_trials": n_trials, "n_train": int(train.sum()), "n_validation": int(val.sum()), "n_test": int(test.sum()), "training_rmse": m_tr["rmse"], "training_r2": m_tr["r2"], "validation_rmse": m_va["rmse"], "validation_r2": m_va["r2"], "test_rmse": m_te["rmse"], "test_r2": m_te["r2"], "seconds": round(time.time() - t0, 1)}
    (RESULTS_DIR / "runs" / f"{rid}.json").write_text(json.dumps(record, indent=2))
    
    if PLOT_TRAIN_VAL:
        plot_train_val(df, train, val, y, yhat, sampling, target, model)
    
    return record


def plot_train_val(df, train, val, y, yhat, sampling, target, model):
    
    dt = pd.to_datetime(df["current_time"], unit="ms", utc=True).dt.tz_convert(TZ)
    
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.plot(dt[train | val], y[train | val], label="Actual", linewidth=1.8)
    ax.plot(dt[train], yhat["train"], label="Predicted (Train)", linewidth=1.8, alpha=0.9)
    ax.plot(dt[val], yhat["validation"], label="Predicted (Val)", linewidth=1.8, alpha=0.9)
    ax.axvline(dt[val].iloc[0], linestyle="--", linewidth=1.2)
    
    for a, b in manifest["night_periods_utc_s"]:
        ax.axvspan(pd.Timestamp(a, unit="s", tz="UTC").tz_convert(TZ), pd.Timestamp(b, unit="s", tz="UTC").tz_convert(TZ), alpha=0.3, color="gray")
        
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b", tz=TZ))
    ax.set_ylabel(target)
    ax.set_title(f"{model} ({sampling}) Actual vs Predicted (Train + Validation)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(IMAGES_PATH / f"{run_id(sampling, target, model)}_train_val.png", dpi=300)
    plt.show()

## TSCV robustness check

Original forward chaining on the first 85% of rows with the hyperparams chosen above. Train rows whose label window reaches the folds val start are dropped. R^2 is not clipped, so a fold whose target barely varies can give a sort of large negative R^2; the pooled R^2 (all fold predictions together) is reported as a steadier summary next to the per-fold values.

In [20]:
def tscv_folds(df, sampling, target):
    sp, cfg = manifest["splits"][sampling], manifest["tscv"]
    n_rows = sp["n_train"] + sp["n_validation"]
    rows_per_h = 3600 / SAMPLING_S[sampling]
    first, val_len, step = (int(cfg[k] * rows_per_h) for k in ("initial_train_h", "val_h", "step_h"))
    min_last = int(cfg["min_last_val_h"] * rows_per_h)
    t, end = df["current_time"].to_numpy(), label_end(df, target)
    has, idx = df[target].notna().to_numpy(), np.arange(len(df))
    folds, train_end = [], first
    
    while train_end < n_rows:
        val_end = min(train_end + val_len, n_rows)
        if val_end - train_end < min_last:
            break
        folds.append(((idx < train_end) & (end <= t[train_end]) & has, (idx >= train_end) & (idx < val_end) & (end <= sp["test_start_ms"]) & has))
        train_end += step
        
    return folds


def run_tscv(df, sampling, target, model, params):
    X, y, t = df[FEATURES].to_numpy(float), df[target].to_numpy(float), df["current_time"].to_numpy()
    iso = lambda ms: pd.Timestamp(int(ms), unit="ms", tz="UTC").isoformat()
    rows = []
    
    for k, (tr, va) in enumerate(tscv_folds(df, sampling, target), 1):
        yhat = build(model, params).fit(X[tr], y[tr]).predict(X[va])
        rows.append({"sampling frequency": sampling, "model_name": model, "target_name": target, "fold": k, "train_start": iso(t[tr].min()), "train_end": iso(t[tr].max()), "val_start": iso(t[va].min()), "val_end": iso(t[va].max()), "n_train": int(tr.sum()), "n_val": int(va.sum()), **metrics(y[va], yhat), "y_true": y[va].tolist(), "y_pred": yhat.tolist()})
    
    return rows


def summarize_tscv(fold_df):
    out = []
    
    for key, g in fold_df.groupby(["sampling frequency", "model_name", "target_name"]):
        row = dict(zip(["sampling frequency", "model_name", "target_name"], key), n_folds=len(g))
        
        for m in ("rmse", "r2"):
            v = g[m].to_numpy(float)
            sd = v.std(ddof=1) if len(v) > 1 else np.nan
            half = stats.t.ppf(0.5 + CI_LEVEL / 2, len(v) - 1) * sd / math.sqrt(len(v)) if len(v) > 1 else np.nan
            row.update({f"mean_{m}": v.mean(), f"median_{m}": np.median(v), f"std_{m}": sd, f"{m}_ci_low": v.mean() - half, f"{m}_ci_high": v.mean() + half})
            
        y_all = np.concatenate([json.loads(s) for s in g["y_true"]])
        p_all = np.concatenate([json.loads(s) for s in g["y_pred"]])
        row.update({f"pooled_{k}": v for k, v in metrics(y_all, p_all).items()})
        out.append(row)
        
    return pd.DataFrame(out)

## Run

Modded this from previous notebook to include the time, as well, since someone (either Dr. H or Dr. W) suggested it in an earlier meeting.

In [9]:
def run(configs):
    
    datasets = {}
    for sampling, target, model in configs:
        rid = run_id(sampling, target, model)
        run_file, tscv_file = RESULTS_DIR / "runs" / f"{rid}.json", RESULTS_DIR / "tscv" / f"{rid}.csv"
        
        if sampling not in datasets:
            datasets[sampling] = load_dataset(sampling)
        df = datasets[sampling]
        
        if not (SKIP_FINISHED and run_file.exists()):
            rec = fit_one(df, sampling, target, model)
            print(f"{rid:60s} val R^2={rec['validation_r2']:.4f}  RMSE={rec['validation_rmse']:.5f}  ({rec['seconds']} s)")
            
        if model in TSCV_MODELS and not (SKIP_FINISHED and tscv_file.exists()):
            params = json.loads(json.loads(run_file.read_text())["params"])
            folds = pd.DataFrame(run_tscv(df, sampling, target, model, params))
            folds["y_true"] = folds["y_true"].map(json.dumps)
            folds["y_pred"] = folds["y_pred"].map(json.dumps)
            folds.to_csv(tscv_file, index=False)


run([(s, t, m) for s in SAMPLINGS for t in TARGETS for m in MODELS_TO_RUN])

30SEC__REXD_FWD_300__dummy_baseline                          val R^2=-0.0165  RMSE=0.06550  (0.1 s)
30SEC__REXD_FWD_300__xgb_baseline                            val R^2=0.2251  RMSE=0.05719  (2.2 s)
30SEC__REXD_FWD_300__xgb_tuned                               val R^2=0.5225  RMSE=0.04489  (137.2 s)
30SEC__REXD_FWD_300__svr_baseline                            val R^2=0.1077  RMSE=0.06136  (5.4 s)
30SEC__REXD_FWD_300__svr_tuned                               val R^2=0.4902  RMSE=0.04638  (333.8 s)
30SEC__REXD_FWD_300__rf_baseline                             val R^2=0.4319  RMSE=0.04896  (7.5 s)
30SEC__REXD_FWD_300__rf_tuned                                val R^2=0.4601  RMSE=0.04774  (224.2 s)
30SEC__REXD_FWD_300_DSAVG__dummy_baseline                    val R^2=-0.0188  RMSE=0.06358  (0.1 s)
30SEC__REXD_FWD_300_DSAVG__xgb_baseline                      val R^2=0.2064  RMSE=0.05611  (1.9 s)
30SEC__REXD_FWD_300_DSAVG__xgb_tuned                         val R^2=0.5414  RMSE=0.04266  (138.1 s)


## Results

Val results drive all comparisons. Test results are only collected into their own file (and are not even looked at until AFTER model select).

In [21]:
runs = pd.DataFrame([json.loads(p.read_text()) for p in sorted((RESULTS_DIR / "runs").glob("*.json"))])
runs["lead_s"] = runs["target_name"].map(lambda t: TARGET_META[t]["lead_s"])
test_cols = ["test_rmse", "test_r2", "n_test"]
runs.drop(columns=test_cols).to_csv(RESULTS_DIR / "validation_results.csv", index=False)
runs[["sampling frequency", "model_name", "target_name", "lead_s"] + test_cols].to_csv(RESULTS_DIR / "test_results.csv", index=False)

tscv_folds_all = pd.concat([pd.read_csv(p) for p in sorted((RESULTS_DIR / "tscv").glob("*.csv"))], ignore_index=True)
tscv_summary = summarize_tscv(tscv_folds_all)
tscv_folds_all.drop(columns=["y_true", "y_pred"]).to_csv(RESULTS_DIR / "tscv_folds.csv", index=False)
tscv_summary.to_csv(RESULTS_DIR / "tscv_summary.csv", index=False)

primary = runs[runs["lead_s"] == 0].drop(columns=test_cols)
primary.sort_values("validation_r2", ascending=False).head(10)

,sampling frequency,model_name,target_name,params,n_trials,n_train,n_validation,training_rmse,training_r2,validation_rmse,validation_r2,seconds,lead_s
303,2MIN,rf_tuned,RSTDEV_FWD_300_SAVGOL,"{""n_estimators"": 200, ""max_depth"": 3}",45,1417,308,0.014467,0.689915,0.013337,0.673172,73.6,0
324,2MIN,rf_tuned,RSTDEV_FWD_600_DSMAX,"{""n_estimators"": 400, ""max_depth"": 3}",45,1441,308,0.020883,0.601508,0.013886,0.646772,74.4,0
296,2MIN,rf_tuned,RSTDEV_FWD_300_DSMAX,"{""n_estimators"": 200, ""max_depth"": 3}",45,1441,308,0.014342,0.631913,0.012297,0.644195,74.8,0
305,2MIN,svr_tuned,RSTDEV_FWD_300_SAVGOL,"{""C"": 2.707972291669852, ""epsilon"": 0.00784847...",100,1417,308,0.017456,0.548535,0.013998,0.639962,31.2,0
302,2MIN,rf_baseline,RSTDEV_FWD_300_SAVGOL,"{""n_estimators"": 500, ""max_depth"": null}",0,1417,308,0.002711,0.989115,0.014076,0.635936,2.2,0
247,2MIN,rf_tuned,REXD_FWD_600_SAVGOL,"{""n_estimators"": 200, ""max_depth"": 3}",45,1417,308,0.079460,0.678257,0.071137,0.634590,72.2,0
295,2MIN,rf_baseline,RSTDEV_FWD_300_DSMAX,"{""n_estimators"": 500, ""max_depth"": null}",0,1441,308,0.004912,0.956833,0.012540,0.630008,2.3,0
135,1MIN,rf_tuned,RSTDEV_FWD_300_SAVGOL,"{""n_estimators"": 1000, ""max_depth"": 4}",45,2831,615,0.013348,0.739752,0.014561,0.624511,112.9,0
240,2MIN,rf_tuned,REXD_FWD_600_DSMAX,"{""n_estimators"": 800, ""max_depth"": 3}",45,1441,308,0.076447,0.622777,0.063446,0.624325,73.2,0
333,2MIN,svr_tuned,RSTDEV_FWD_600_SAVGOL,"{""C"": 7.9345731137131486, ""epsilon"": 0.0263603...",100,1417,308,0.028511,0.434850,0.016378,0.624031,32.2,0


In [22]:
tscv_summary.sort_values("pooled_r2", ascending=False).head(10)

,sampling frequency,model_name,target_name,n_folds,mean_rmse,median_rmse,std_rmse,rmse_ci_low,rmse_ci_high,mean_r2,median_r2,std_r2,r2_ci_low,r2_ci_high,pooled_rmse,pooled_r2
126,1MIN,xgb_tuned,REXD_FWD_300_DSMAX,3,0.055165,0.053226,0.004246,0.044618,0.065713,0.441520,0.447857,0.136694,0.101953,0.781088,0.055421,0.505970
281,2MIN,xgb_tuned,RSTDEV_FWD_300_DSAVG,3,0.009167,0.009342,0.000304,0.008411,0.009923,0.398770,0.449391,0.159143,0.003438,0.794103,0.009187,0.465860
414,30SEC,xgb_tuned,REXD_FWD_300_DSMAX,3,0.053679,0.053593,0.003798,0.044244,0.063115,0.384269,0.423463,0.206081,-0.127664,0.896202,0.053952,0.465298
129,1MIN,xgb_tuned,REXD_FWD_600_DSAVG,3,0.059562,0.061403,0.003225,0.051551,0.067573,0.412000,0.417191,0.105908,0.148910,0.675089,0.059801,0.454313
138,1MIN,xgb_tuned,RSTDEV_FWD_300_DSMAX,3,0.014100,0.014509,0.001611,0.010097,0.018103,0.385622,0.384878,0.179759,-0.060924,0.832168,0.014246,0.441466
125,1MIN,xgb_tuned,REXD_FWD_300_DSAVG,3,0.044349,0.042684,0.003250,0.036275,0.052423,0.347343,0.355471,0.195345,-0.137920,0.832605,0.044533,0.436740
137,1MIN,xgb_tuned,RSTDEV_FWD_300_DSAVG,3,0.009699,0.009933,0.000732,0.007882,0.011517,0.367422,0.427178,0.204870,-0.141503,0.876348,0.009757,0.434512
174,2MIN,rf_tuned,REXD_FWD_300_DSMAX,3,0.064341,0.067694,0.007586,0.045497,0.083186,0.350200,0.364782,0.244165,-0.256340,0.956741,0.065043,0.428142
282,2MIN,xgb_tuned,RSTDEV_FWD_300_DSMAX,3,0.015940,0.017261,0.002379,0.010031,0.021849,0.366576,0.373436,0.226384,-0.195792,0.928944,0.016184,0.422822
417,30SEC,xgb_tuned,REXD_FWD_600_DSAVG,3,0.061984,0.064747,0.006909,0.044820,0.079147,0.382973,0.337506,0.153307,0.002139,0.763808,0.062617,0.422147


## Preprocessing factor tests (Table 5)

Paired tests on val metrics: Wilcoxon for two levels, Friedman for more. Blocks are the combinations of the other factors and the model. NN results from colab are included when results/nn/validation_results.csv is present.

In [23]:
FACTOR_ALPHA = 0.05
NN_RESULTS = Path("results/nn/validation_results.csv")

fac = pd.concat([primary] + ([pd.read_csv(NN_RESULTS)] if NN_RESULTS.exists() else []), ignore_index=True)
fac = fac[fac["model_name"] != "dummy_baseline"].copy()
for key in ("metric", "window_samples", "variant", "lead_s"):
    fac[key] = fac["target_name"].map(lambda t, k=key: TARGET_META[t][k])
fac = fac[fac["lead_s"] == 0]

FACTORS = {"sampling frequency": ["metric", "window_samples", "variant"], "window_samples": ["sampling frequency", "metric", "variant"], "variant": ["sampling frequency", "metric", "window_samples"], "metric": ["sampling frequency", "window_samples", "variant"]}

rows = []
for factor, others in FACTORS.items():
    
    for value in ("validation_r2", "validation_rmse"):
        wide = fac.pivot_table(index=["model_name"] + others, columns=factor, values=value, aggfunc="mean").dropna()
        
        if wide.shape[0] < 2 or wide.shape[1] < 2:
            continue
            
        arrays = [wide[c].to_numpy(float) for c in wide.columns]
        
        if len(arrays) == 2:
            stat, p = (0.0, 1.0) if np.allclose(arrays[0], arrays[1]) else stats.wilcoxon(arrays[0], arrays[1])
            test = "Wilcoxon"
        else:
            stat, p = stats.friedmanchisquare(*arrays)
            test = "Friedman"
            
        rows.append({"factor": factor, "metric": value, "test": test, "levels": list(map(str, wide.columns)), "n_blocks": wide.shape[0], "statistic": float(stat), "p_value": float(p), "significant": bool(p < FACTOR_ALPHA)})

factor_tests = pd.DataFrame(rows)
factor_tests.to_csv(RESULTS_DIR / "preprocessing_factor_tests.csv", index=False)
factor_tests

,factor,metric,test,levels,n_blocks,statistic,p_value,significant
0,sampling frequency,validation_r2,Friedman,"[1MIN, 2MIN, 30SEC]",384,129.390625,8.001709e-29,True
1,sampling frequency,validation_rmse,Friedman,"[1MIN, 2MIN, 30SEC]",384,65.333333,6.501999e-15,True
2,window_samples,validation_r2,Friedman,"[300, 600, 1200]",384,120.161458,8.077386e-27,True
3,window_samples,validation_rmse,Friedman,"[300, 600, 1200]",384,688.380208,3.312356e-150,True
4,variant,validation_r2,Friedman,"[DEC, DSAVG, DSMAX, SAVGOL]",288,34.241667,1.761584e-07,True
5,variant,validation_rmse,Friedman,"[DEC, DSAVG, DSMAX, SAVGOL]",288,658.395833,2.202942e-142,True
6,metric,validation_r2,Wilcoxon,"[REXD, RSTDEV]",576,62678.000000,3.259123e-07,True
7,metric,validation_rmse,Wilcoxon,"[REXD, RSTDEV]",576,2292.000000,6.525061e-91,True


## Lead-time runs (L = 2 / 5 / 10 min) (NOT USED YET -- a future work thing)

Same window, preprocessing and model as the selected pipeline, with the label window starting L after t. With LEAD_TIME_PIPELINES = "locked", rerun this notebook after STATIC_SIMULATION has locked a config (finished runs are skipped).

In [ ]:
def lead_target(target, lead_s):
    base = TARGET_META[target]["base"]
    return next((t for t, m in TARGET_META.items() if m["base"] == base and m["lead_s"] == lead_s), None)


if RUN_LEAD_TIMES:
    
    if LEAD_TIME_PIPELINES == "locked":
        pipelines = [json.loads(LOCKED_SELECTION_PATH.read_text())] if LOCKED_SELECTION_PATH.exists() else []
        
        if not pipelines:
            print("No locked selection yet - skipping lead-time runs.")
    else:
        pipelines = LEAD_TIME_PIPELINES
        
    configs = []
    for p in pipelines:
        
        if p["model"] not in MODELS_TO_RUN:
            print(f"{p['model']} is trained in the Colab notebook; run its lead-time targets there.")
            continue
            
        configs += [(p["sampling"], lead_target(p["target"], L), p["model"]) for L in LEAD_TIMES_S if lead_target(p["target"], L)]
        
    run(configs)